# Test Complete Chat Workflow Pipeline

This notebook tests the end-to-end chat service workflow:
1. Intent Router → Routes to appropriate agent
2. Agent executes → Generates response
3. Complete conversation flow

In [1]:
from sahiloan_chatbot.application.chat_service.workflow.graph import create_chat_graph
from sahiloan_chatbot.application.chat_service.workflow.state import ChatState
from sahiloan_chatbot.infrastructure.db import get_pinecone_index
import json
import time

/Users/vishnum/Library/Caches/pypoetry/virtualenvs/sahiloan-chatbot-h6twZ8gO-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-26 18:12:51.722 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:_load_intent_eval_dataset:64 - Looking for eval dataset at: /Users/vishnum/sahiloan-customer-chatbot/data/evals/intent_router.json
2026-01-26 18:12:51.724 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:_load_intent_eval_dataset:77 - Loading 25 eval queries for intent caching...
2026-01-26 18:13:01.672 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:_load_intent_eval_dataset:89 - ✅ Cached 25 intent router embeddings


In [2]:
print("Initializing chat workflow graph...")
graph_builder = create_chat_graph()
graph = graph_builder.compile()
print(f"✅ Graph compiled successfully")
print(f"✅ Available nodes: intent_router, general_agent, loan_agent, document_agent")
print(f"✅ Pinecone connected")

Initializing chat workflow graph...
✅ Graph compiled successfully
✅ Available nodes: intent_router, general_agent, loan_agent, document_agent
✅ Pinecone connected


## Helper Function: Execute Complete Pipeline

In [3]:
def execute_pipeline(query: str, verbose: bool = True):
    """
    Execute the complete pipeline for a query using the LangGraph workflow.
    
    Args:
        query: User's input query
        verbose: Whether to print detailed logs
        
    Returns:
        Final state with response
    """
    if verbose:
        print(f"\n{'='*80}")
        print(f"🔍 QUERY: {query}")
        print(f"{'='*80}")
    
    start_time = time.time()
    
    # Step 1: Create initial state
    initial_state = ChatState(
        messages=[{"role": "user", "content": query}]
    )
    
    if verbose:
        print(f"\n🚀 EXECUTING WORKFLOW GRAPH...")
    
    # Step 2: Execute the entire graph (handles routing and agent execution automatically)
    final_state = graph.invoke(initial_state)
    
    # Extract results
    route = final_state.get("route_to", "unknown")
    
    # Extract response (handle both dict and Message objects)
    if len(final_state["messages"]) > 1:
        last_message = final_state["messages"][-1]
        if isinstance(last_message, dict):
            response = last_message["content"]
        else:
            # It's a LangChain Message object (AIMessage, etc.)
            response = last_message.content
    else:
        response = None
    
    total_time = (time.time() - start_time) * 1000
    
    if verbose:
        print(f"   ✅ Route determined: {route}")
        print(f"   ✅ Graph execution completed")
        
        print(f"\n📝 RESPONSE:")
        print(f"{'─'*80}")
        if response:
            print(response)
        else:
            print("(No response generated - conversation ended)")
        print(f"{'─'*80}")
        print(f"\n⏱️ Total pipeline time: {total_time:.2f}ms")
        print(f"{'='*80}\n")
    
    return {
        "query": query,
        "route": route,
        "response": response,
        "latency_ms": total_time,
        "state": final_state
    }

## Test 1: Single Query (Detailed)

In [5]:
# Test with a single query showing full details
result = execute_pipeline("")

2026-01-26 18:14:36.360 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:152 - intent_router_started
2026-01-26 18:14:36.361 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:156 - intent_router_user_input: What is sahiloan



🔍 QUERY: What is sahiloan

🚀 EXECUTING WORKFLOW GRAPH...


2026-01-26 18:14:36.826 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:_find_similar_intent:142 - No semantic match | best similarity: 0.6793 (< 0.8)
2026-01-26 18:14:36.826 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:172 - No semantic match, using LLM for intent classification
2026-01-26 18:14:37.615 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:186 - intent_router_llm_response: general_agent
2026-01-26 18:14:37.616 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:intent_router:197 - intent_router_completed | route: general_agent | method: llm | latency: 1256.22ms | llm: gpt-4o-mini
2026-01-26 18:14:37.618 | INFO     | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:213 - general_agent_started
2026-01-26 18:14:37.619 | DEBUG    | sahiloan_chatbot.application.chat_service.workflow.nodes:general_agent:217 - general_agent_query: What is sahiloan

   ✅ Route determined: end
   ✅ Graph execution completed

📝 RESPONSE:
────────────────────────────────────────────────────────────────────────────────
Sahiloan is a loan advisory platform designed to assist individuals in finding the right loan options for their needs. We provide guidance and support throughout the loan process, helping customers understand their choices and make informed decisions. If you have specific questions about our services or how we can assist you, feel free to ask!
────────────────────────────────────────────────────────────────────────────────

⏱️ Total pipeline time: 3996.58ms



## Test 2: Multiple Queries (General Agent)

In [ ]:
# Test queries that should route to general_agent
general_queries = [
    "What is a home loan?",
    "Is Sahiloan free to use?",
    "How is EMI calculated?",
    "What credit score do I need?",
    "What types of loans does Sahiloan handle?"
]

print("="*80)
print("TESTING GENERAL AGENT QUERIES")
print("="*80)

results = []
for query in general_queries:
    result = execute_pipeline(query, verbose=False)
    results.append(result)
    
    print(f"\n📝 Query: {query}")
    print(f"   Route: {result['route']}")
    print(f"   Response: {result['response'][:150]}..." if result['response'] else "   Response: None")
    print(f"   Latency: {result['latency_ms']:.2f}ms")

## Test 3: Mixed Intent Queries

In [ ]:
# Test queries with different expected routes
mixed_queries = [
    ("What is the difference between home loan and LAP?", "general_agent"),
    ("How many EMIs have I paid on my SBI loan?", "loan_agent"),
    ("Please verify my salary slip document", "document_agent"),
    ("Thanks, that's all I needed!", "end"),
]

print("="*80)
print("TESTING MIXED INTENT QUERIES")
print("="*80)

for query, expected_route in mixed_queries:
    result = execute_pipeline(query, verbose=False)
    
    match_status = "✅" if result['route'] == expected_route else "❌"
    
    print(f"\n{match_status} Query: {query}")
    print(f"   Expected: {expected_route}")
    print(f"   Got: {result['route']}")
    if result['response']:
        print(f"   Response: {result['response'][:100]}...")
    print(f"   Latency: {result['latency_ms']:.2f}ms")

## Test 4: Interactive Testing (Custom Queries)

In [ ]:
# Change this query to test with your own inputs
custom_query = "Can Sahiloan help me reduce my interest rate?"

print("\n" + "="*80)
print("CUSTOM QUERY TEST")
print("="*80)

result = execute_pipeline(custom_query, verbose=True)

## Test 5: Performance Analysis

In [ ]:
# Performance analysis across multiple queries
test_queries = [
    "What is Sahiloan?",
    "How does EMI work?",
    "Tell me about loan eligibility",
    "Is there any processing fee?",
    "What documents are needed?",
]

print("="*80)
print("PERFORMANCE ANALYSIS")
print("="*80)

latencies = []
routes_count = {}

for query in test_queries:
    result = execute_pipeline(query, verbose=False)
    latencies.append(result['latency_ms'])
    
    route = result['route']
    routes_count[route] = routes_count.get(route, 0) + 1

# Statistics
avg_latency = sum(latencies) / len(latencies)
min_latency = min(latencies)
max_latency = max(latencies)

print(f"\n📊 Statistics:")
print(f"   Total queries: {len(test_queries)}")
print(f"   Average latency: {avg_latency:.2f}ms")
print(f"   Min latency: {min_latency:.2f}ms")
print(f"   Max latency: {max_latency:.2f}ms")

print(f"\n📈 Route Distribution:")
for route, count in routes_count.items():
    percentage = (count / len(test_queries)) * 100
    print(f"   {route}: {count} ({percentage:.1f}%)")

print(f"\n✅ All tests completed!")
print("="*80)

## Test 6: Conversation Simulation (Multi-Turn)

In [ ]:
# Simulate a multi-turn conversation
conversation = [
    "Hi, what is Sahiloan?",
    "Is your service free?",
    "What types of loans do you handle?",
    "How is EMI calculated?",
    "Thanks, that helps!",
]

print("="*80)
print("CONVERSATION SIMULATION")
print("="*80)

for turn, query in enumerate(conversation, 1):
    print(f"\n{'─'*80}")
    print(f"Turn {turn}")
    print(f"{'─'*80}")
    
    result = execute_pipeline(query, verbose=False)
    
    print(f"👤 User: {query}")
    print(f"🤖 Bot (via {result['route']}): {result['response'][:200]}..." if result['response'] else f"🤖 Bot: (No response)")
    
    if result['route'] == 'end':
        print(f"\n✅ Conversation ended after {turn} turns")
        break

print(f"\n{'='*80}")

## Test 7: Edge Cases

In [ ]:
# Test edge cases
edge_cases = [
    "",  # Empty query
    "hello",  # Simple greeting
    "asdfghjkl",  # Gibberish
    "What's the weather today?",  # Unrelated question
    "Tell me about cryptocurrency loans",  # Out of scope
]

print("="*80)
print("EDGE CASE TESTING")
print("="*80)

for query in edge_cases:
    display_query = query if query else "[EMPTY]"
    print(f"\n🧪 Testing: {display_query}")
    
    try:
        result = execute_pipeline(query, verbose=False)
        print(f"   Route: {result['route']}")
        if result['response']:
            print(f"   Response: {result['response'][:100]}...")
        print(f"   Status: ✅ Handled")
    except Exception as e:
        print(f"   Status: ❌ Error - {str(e)}")

print(f"\n{'='*80}")